In [1]:
%load_ext autoreload
%autoreload 2

In [112]:
import os
import sys
from pathlib import Path
import glob

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

from src.load_data import load_confocal, load_confocal2

# Experimental conditions

In [113]:
experimental_path = Path.cwd().parent / 'data/silver_layer/experimental_results.pkl'
df_experimental = pd.read_pickle(experimental_path)

# Confocal

In [98]:
data_path = Path.cwd().parent / 'data/bronze_layer/confocal'

In [83]:
z1 = load_confocal(data_path, '2024_02_27', 12)
z2 = load_confocal(data_path, '2024_02_28', 12)
z3 = load_confocal(data_path, '2024_03_04', 12)
z4 = load_confocal(data_path, '2024_03_05', 12)
z5 = load_confocal(data_path, '2024_03_06', 12)
z6 = load_confocal(data_path, '2024_03_12', 12)
z7 = load_confocal(data_path, '2024_03_13', 12)
z8 = load_confocal(data_path, '2024_03_14', 12)
z9 = load_confocal(data_path, '2024_03_16', 14)
z10 = load_confocal(data_path, '2024_03_18', 14)
z11 = load_confocal(data_path, '2024_03_19', 14)
z12 = load_confocal(data_path, '2024_03_21', 14)
z13 = load_confocal(data_path, '2024_03_22', 14)
z14 = load_confocal(data_path, '2024_03_22_v2', 14)
z15 = load_confocal(data_path, '2024_03_23', 14)
z16 = load_confocal(data_path, '2024_03_25', 14)
z17 = load_confocal(data_path, '2024_03_25_v2', 14)
z18 = load_confocal(data_path, '2024_03_25_v3', 14)
z19 = load_confocal(data_path, '2024_03_26', 14)

In [84]:
z = []

z.extend([z1, z2, z3, z4, z5, z6, z7, z8, z9, z10, z11, z12, z13, z14, z15, z16, z17, z18, z19])

In [85]:
columns = ['t1_mean', 't1_median', 't1_std', 't1_missing_percentage', 't2_mean', 't2_median', 't2_std', 't2_missing_percentage', 'df', 'number', 'delta_time']

In [86]:
rows = []

run_id = 1

for j, zn in enumerate(z):

    for i, run_df in enumerate(zn):

        delta_time = (
            run_df['time'].diff().mean()
            if len(run_df) > 1 else np.nan
        )

        rows.append({

            'run_id': run_id,

            't1_mean': round(run_df['t1'].mean(), 3),
            't1_median': round(run_df['t1'].median(), 3),
            't1_std': round(run_df['t1'].std(), 3),

            't1_missing_percentage':
                round(
                    100 - run_df['t1'].count()
                    / run_df['time'].count() * 100,
                    1
                ),

            't2_mean': round(run_df['t2'].mean(), 3),
            't2_median': round(run_df['t2'].median(), 3),
            't2_std': round(run_df['t2'].std(), 3),

            't2_missing_percentage':
                round(
                    100 - run_df['t2'].count()
                    / run_df['time'].count() * 100,
                    1
                ),

            'delta_time': delta_time,
        })

        run_id += 1

df = pd.DataFrame(rows)

df['run_id'] = df['run_id'].astype(int)

In [87]:
save_path = Path.cwd().parent / 'data/silver_layer'
save_path.mkdir(parents=True, exist_ok=True)

df.to_pickle(
    save_path / 'confocal_results.pkl',
)

df.to_csv(
    save_path / 'confocal_results.csv'
)

In [88]:
save_path = Path.cwd().parent / 'data/silver_layer/confocal_runs'

save_path.mkdir(exist_ok=True)

run_id = 1

for zn in z:

    for run_df in zn:

        filename = save_path / f"confocal_run_{run_id}.csv"

        run_df.to_csv(filename, index=False)

        run_id += 1

# Process tube thickness

In [104]:
confocal_path = (
    Path.cwd().parent
    / "data/silver_layer/confocal_runs_processed"
)

rows = []

# Sort by run number if files are named confocal_run_1.csv, confocal_run_2.csv, ...
files = sorted(
    confocal_path.glob("confocal_run_*.csv"),
    key=lambda x: int(x.stem.split("_")[-1])
)

for run_id, file in enumerate(files, start=1):

    run_df = pd.read_csv(file)

    delta_time = (
        run_df["time"].diff().mean()
        if len(run_df) > 1 else np.nan
    )

    rows.append({

        "run_id": run_id,

        "t1_mean": round(run_df["t1"].mean(), 3),
        "t1_median": round(run_df["t1"].median(), 3),
        "t1_std": round(run_df["t1"].std(), 3),

        "t1_missing_percentage":
            round(
                100
                - run_df["t1"].count()
                / run_df["time"].count() * 100,
                1
            ),

        "t2_mean": round(run_df["t2"].mean(), 3),
        "t2_median": round(run_df["t2"].median(), 3),
        "t2_std": round(run_df["t2"].std(), 3),

        "t2_missing_percentage":
            round(
                100
                - run_df["t2"].count()
                / run_df["time"].count() * 100,
                1
            ),

        "delta_time": delta_time,
    })

df = pd.DataFrame(rows)

df["run_id"] = df["run_id"].astype(int)

In [105]:
save_path = Path.cwd().parent / 'data/silver_layer'
save_path.mkdir(parents=True, exist_ok=True)

df.to_pickle(
    save_path / 'confocal_results_processed.pkl',
)

df.to_csv(
    save_path / 'confocal_results_processed.csv'
)

# Merge experimental

In [109]:
df_combined = pd.merge(df, df_experimental[['run_id', 'jl', 'jg', 'date_identifier', 'pattern', 'reduced_pattern']], on='run_id', how='left')

In [110]:
pd.read_pickle(save_path/'confocal_results_processed.pkl')['t2_mean'].sum()

np.float64(346.842)

In [111]:
pd.read_pickle(save_path/'confocal_results.pkl')['t2_mean'].sum()

np.float64(347.508)